# v4.2 Risk-Increase Confirmation Ablation

This notebook follows the state-2 tail diagnostic. It separates:

- a **mechanical one-session execution delay**;
- one-session confirmation on the `0→1` bridge entry;
- one-session confirmation on the `1→2` leverage entry;
- confirmation on both risk-increasing transitions.

No price, VIX, VXN, allocation, TQQQ weight, or official cost parameter is changed.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("artifacts/evidence/qqqi_qqq_tqqq_v4_2_risk_confirmation_v4_3_research")
summary = json.loads((ROOT / "confirmation_summary.json").read_text(encoding="utf-8"))
metrics = pd.read_csv(ROOT / "confirmation_metrics.csv", index_col=0)
segments = pd.read_csv(ROOT / "confirmation_chronological_segments.csv")
events = pd.read_csv(ROOT / "confirmation_difference_events.csv", parse_dates=["start_date", "end_date"])
event_summary = pd.read_csv(ROOT / "confirmation_event_summary.csv")

summary["economic_sample"], summary["research_gate"]

## 1. Full-sample comparison

In [ ]:
metrics[[
    "total_return", "cagr", "annual_volatility", "sharpe", "sortino",
    "max_drawdown", "calmar", "turnover_units", "state_2_sessions",
    "cagr_delta_vs_baseline", "max_drawdown_delta_vs_baseline",
]]

In [ ]:
focus = metrics.loc[[
    "baseline", "fixed_execution_delay_1",
    "bridge_entry_confirmation_1", "leverage_entry_confirmation_1",
    "risk_increase_confirmation_1",
]]
plt.figure(figsize=(10, 6))
plt.scatter(focus["max_drawdown"] * 100, focus["cagr"] * 100)
for name, row in focus.iterrows():
    plt.annotate(name, (row["max_drawdown"] * 100, row["cagr"] * 100))
plt.xlabel("Maximum drawdown (%)")
plt.ylabel("CAGR (%)")
plt.title("Timing ablations versus current v4.2")
plt.tight_layout()
plt.show()

## 2. Chronological stability

In [ ]:
segments.pivot_table(
    index="strategy",
    columns="segment",
    values=["cagr", "sharpe", "max_drawdown", "calmar"],
)

## 3. Event attribution and concentration

In [ ]:
event_summary

In [ ]:
events.sort_values(
    ["scenario", "net_return_delta"], ascending=[True, False]
).head(40)

## 4. Research gate

In [ ]:
gate = summary["research_gate"]
pd.Series(gate["gates"], name="passed").to_frame().join(
    pd.Series(gate["measured"], name="measured")
)

In [ ]:
print("Retrospective gate passed:", gate["passes_retrospective_research_gate"])
print("Direct promotion authorized:", gate["promotion_authorized"])
print("Next direction:", gate["next_direction"])